In [ ]:
import os
import pandas as pd
path = os.path.join( "..", "data", "processed", "accidents_clean.csv")
accidents_clean = pd.read_csv(path)

In [ ]:
import numpy as np 

taille_cellule = 0.05  # ~5 km

accidents_clean = accidents_clean.dropna(subset=['lat', 'long']).copy()

accidents_clean['cell_x'] = (accidents_clean['long'] // taille_cellule).astype(int)
accidents_clean['cell_y'] = (accidents_clean['lat'] // taille_cellule).astype(int)

grille = (
    accidents_clean
    .groupby(['cell_x', 'cell_y'])
    .agg(
        nb_accidents=('grav', 'count'),
        grav_mediane=('grav', 'median'),
        lat_centre=('lat', 'mean'),
        long_centre=('long', 'mean')
    )
    .reset_index()
)

couleurs = {
    1: 'green',
    2: 'orange',
    3: 'red',
    4: 'darkred'
}

import folium

m = folium.Map(location=[46.6, 2.2], zoom_start=6)

for _, row in grille.iterrows():
    grav = int(row['grav_mediane'])
    couleur = couleurs.get(grav, 'gray')
    
    folium.Circle(
        location=[row['lat_centre'], row['long_centre']],
        # radius = np.sqrt(row['nb_accidents']) * (grav),
        radius= (row['nb_accidents'])*(np.exp(((grav-1)/2)**(grav+1))),  # rayon dépendant du nombre d'accidents pondéré par la gravité
        color=couleur,
        fill=True,
        fill_color=couleur,
        fill_opacity=0.6,
        popup=f"{row['nb_accidents']} accidents<br>Gravité médiane : {grav}"
    ).add_to(m)

m

In [ ]:
geo_departements = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/departements-version-simplifiee.geojson"

import geopandas as gpd
from shapely.geometry import Point

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

# Charger le fond géographique
gdf_departements = gpd.read_file(geo_departements)

# Créer un GeoDataFrame à partir des accidents
gdf_accidents = gpd.GeoDataFrame(
    accidents_clean.dropna(subset=['lat', 'long']).copy(),
    geometry=gpd.points_from_xy(accidents_clean['long'], accidents_clean['lat']),
    crs="EPSG:4326"
)

# Jointure spatiale : chaque point reçoit le code du département
gdf_accidents = gpd.sjoin(gdf_accidents, gdf_departements[['code', 'nom', 'geometry']], how="left", predicate="within")

df_stats = (
    gdf_accidents
    .groupby('code')  # code du département
    .agg(
        nb_accidents=('grav', 'count'),
        grav_moyenne=('grav', 'mean')
    )
    .reset_index()
)

import folium

m = folium.Map(location=[46.6, 2.2], zoom_start=6)

folium.Choropleth(
    geo_data=geo_departements,
    data=df_stats,
    columns=["code", "grav_moyenne"],
    key_on="feature.properties.code",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Gravité moyenne des accidents par département"
).add_to(m)

m.save("../reports/figures/1.0-Seb-Map_departement.html")

options = Options()
options.headless = True
driver = webdriver.Chrome(options=options)

# Charger la carte
driver.set_window_size(1024, 768)  
driver.get("file://" + os.path.abspath("../reports/figures/1.0-Seb-Map_departement.html"))
time.sleep(3)  

# Capture écran
driver.save_screenshot("../reports/figures/1.0-Seb-Map_departement.png")
driver.quit()

m

In [ ]:
import folium
import geopandas as gpd

# 1. Charger les communes
geo_communes = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/communes-avec-outre-mer.geojson"
gdf_communes = gpd.read_file(geo_communes)

# 2. Créer un GeoDataFrame pour les accidents
gdf_accidents = gpd.GeoDataFrame(
    accidents_clean.dropna(subset=['lat', 'long']).copy(),
    geometry=gpd.points_from_xy(accidents_clean['long'], accidents_clean['lat']),
    crs="EPSG:4326"
)

# 3. Jointure spatiale : associer chaque accident à une commune
gdf_accidents = gpd.sjoin(
    gdf_accidents,
    gdf_communes[['code', 'nom', 'geometry']],
    how="left",
    predicate="within"
)

# 4. Statistiques par commune
df_stats = (
    gdf_accidents
    .groupby('code')
    .agg(
        nb_accidents=('grav', 'count'),
        grav_moyenne=('grav', 'mean')
    )
    .reset_index()
)

# 5. Fusionner avec le GeoDataFrame des communes
gdf_communes = gdf_communes.merge(df_stats, on="code", how="left")

# 6. Créer la carte
m = folium.Map(location=[46.6, 2.2], zoom_start=6)

# 7. Couleur par gravité moyenne
folium.Choropleth(
    geo_data=gdf_communes,
    data=df_stats,
    columns=["code", "grav_moyenne"],
    key_on="feature.properties.code",
    fill_color="YlOrRd",
    fill_opacity=0.6,
    line_opacity=0.3,
    nan_fill_color="white",
    legend_name="Gravité moyenne des accidents par commune"
).add_to(m)

# 8. Popups interactifs
folium.GeoJson(
    gdf_communes,
    name="Communes",
    style_function=lambda feature: {
        "fillOpacity": 0,
        "weight": 0,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["nom", "nb_accidents", "grav_moyenne"],
        aliases=["Commune :", "Accidents :", "Gravité moyenne :"],
        localize=True,
        sticky=True
    )
).add_to(m)

m
